# Manual RAG Pipeline: Mechanisms First

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline from scratch.
You'll see every step explicitly before we move to frameworks like LangChain.

**Works on:** Google Colab, Local Jupyter (Mac/Windows/Linux)

**Pipeline Overview:**
```
Documents → Chunking → Embedding → Index (FAISS)
                                        ↓
User Query → Embed Query → Similarity Search → Top-K Chunks
                                                    ↓
                                        Prompt Assembly → LLM → Answer
```

## TODO — Topic 5 RAG Course Project Checklist

- **Exercise 0:** Set-up — Get notebook running; unzip Corpora.zip. Use PDFs from `Corpora/<corpus>/pdf_embedded/`.
- **Exercise 1:** Open model RAG vs no RAG — Compare Qwen 2.5 1.5B with/without RAG on Model T manual and Congressional Record.
- **Exercise 2:** Open model + RAG vs large model — Run GPT-4o Mini with no tools on same queries.
- **Exercise 3:** Open model + RAG vs frontier chat — Compare local Qwen+RAG vs GPT-4/Claude (web).
- **Exercise 4:** Effect of top-K — Test k = 1, 3, 5, 10, 20.
- **Exercise 5:** Unanswerable questions — Off-topic, related-but-missing, false premise.
- **Exercise 6:** Query phrasing sensitivity — Same question in 5+ phrasings.
- **Exercise 7:** Chunk overlap — Re-chunk with overlap 0, 64, 128, 256.
- **Exercise 8:** Chunk size — Chunk at 128, 256, 512, 1024, 2048.
- **Exercise 9:** Retrieval score analysis — 10 queries, top-10 chunks, score distribution.
- **Exercise 10:** Prompt template variations — Minimal, strict grounding, citation, permissive, structured.
- **Exercise 11:** Cross-document synthesis — Questions needing multiple chunks.

## Setup

First, let's install the required packages and detect our compute environment.

In [95]:
# Install dependencies
# On Colab, these install quickly. Locally, you may already have them.
# Use a kernel-aware install when available; fall back to subprocess otherwise.
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate ipyfilechooser')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate', 'ipyfilechooser'])
# For Exercise 2 (GPT-4o Mini): add 'openai' to the list above if needed


Note: you may need to restart the kernel to use updated packages.


In [96]:
# =============================================================================
# ENVIRONMENT AND DEVICE DETECTION
# =============================================================================
import os
import sys

# Enable MPS fallback for any PyTorch operations not yet implemented on Metal
# This MUST be set before importing torch
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Prevent kernel crash from duplicate OpenMP libraries (PyTorch + FAISS conflict on macOS)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import Tuple

def detect_environment() -> str:
    """Detect if we're running on Colab or locally."""
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device() -> Tuple[str, torch.dtype]:
    """
    Detect the best available compute device.
    
    Priority: CUDA > MPS (Apple Silicon) > CPU
    
    Returns:
        Tuple of (device_string, recommended_dtype)
        
    Notes:
        - CUDA: Use float16 for memory efficiency (Tensor Cores optimize this)
        - MPS: Use float32 - Apple Silicon doesn't have the same float16 
               optimizations as NVIDIA, and float32 is often faster
        - CPU: Use float32 (float16 not well supported on CPU)
    """
    if torch.cuda.is_available():
        device = 'cuda'
        dtype = torch.float16
        device_name = torch.cuda.get_device_name(0)
        memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ Using CUDA GPU: {device_name} ({memory_gb:.1f} GB)")
        
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = 'mps'
        dtype = torch.float32  # float32 is often faster on Apple Silicon!
        print("✓ Using Apple Silicon GPU (MPS)")
        print("  Note: Using float32 (faster than float16 on Apple Silicon)")
        
    else:
        device = 'cpu'
        dtype = torch.float32
        print("⚠ Using CPU (no GPU detected)")
        print("  Tip: For faster processing, use a machine with a GPU")
    
    return device, dtype

# Detect environment and device
ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()

print(f"\nEnvironment: {ENVIRONMENT.upper()}")
print(f"Device: {DEVICE}, Dtype: {DTYPE}")

✓ Using CUDA GPU: NVIDIA RTX A6000 (51.0 GB)

Environment: LOCAL
Device: cuda, Dtype: torch.float16


## Load Your Documents

**Cell 1:** Configure your document source and select/upload files
- **Local Jupyter**: Use the folder picker, then run Cell 2
- **Colab + Upload**: Files upload immediately (blocking), then run Cell 2
- **Colab + Drive**: Set `USE_GOOGLE_DRIVE = True`, mounts Drive and shows picker, then run Cell 2

**Cell 2:** Confirms selection and lists documents

In [97]:
# =============================================================================
# CELL 1: SELECT DOCUMENT SOURCE
# =============================================================================
# This cell either:
#   - Shows a folder picker (Local or Colab+Drive) - NON-BLOCKING
#   - Shows an upload dialog (Colab+Upload) - BLOCKING
#
# If a folder picker is shown, SELECT YOUR FOLDER BEFORE running Cell 2.
# The picker widget is non-blocking, so the code continues before you select.
# =============================================================================

from pathlib import Path

# ------------- COLAB USERS: CONFIGURE HERE -------------
USE_GOOGLE_DRIVE = False  # Set to True to use Google Drive instead of uploading
# -------------------------------------------------------

# Default folder: use Corpora from course project (unzip Corpora.zip first).
_folder_default = Path("Corpora/ModelTService")
DOC_FOLDER = str(_folder_default) if _folder_default.exists() else "documents"
folder_chooser = None  # Will hold the picker widget if used

if ENVIRONMENT == 'colab':
    if USE_GOOGLE_DRIVE:
        # ----- COLAB + GOOGLE DRIVE -----
        # Mount Drive first, then show folder picker
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
        print("✓ Google Drive mounted\n")
        
        # Now show folder picker for the Drive
        try:
            from ipyfilechooser import FileChooser
            
            folder_chooser = FileChooser(
                path='/content/drive/MyDrive',
                title='Select your documents folder in Google Drive',
                show_only_dirs=True,
                select_default=True
            )
            print("📁 Select your documents folder below, then run Cell 2:")
            print("   (The picker is non-blocking - select BEFORE running the next cell)")
            display(folder_chooser)
            
        except ImportError:
            # Fallback: manual path entry
            print("Folder picker not available.")
            print("Edit DOC_FOLDER below with your Google Drive path, then run Cell 2:")
            DOC_FOLDER = '/content/drive/MyDrive/your_documents_folder'  # ← Edit this!
            print(f"  DOC_FOLDER = '{DOC_FOLDER}'")
    else:
        # ----- COLAB + UPLOAD -----
        # Upload dialog blocks until complete, so DOC_FOLDER is ready when done
        from google.colab import files
        os.makedirs(DOC_FOLDER, exist_ok=True)
        
        print("Upload your documents (PDF, TXT, or MD):")
        print("(This dialog blocks until upload is complete)\n")
        uploaded = files.upload()
        
        for filename in uploaded.keys():
            os.rename(filename, f'{DOC_FOLDER}/{filename}')
            print(f"  ✓ Saved: {DOC_FOLDER}/{filename}")
        
        print(f"\n✓ Upload complete. Run Cell 2 to continue.")

else:
    # ----- LOCAL JUPYTER -----
    # Show folder picker
    print("Running locally\n")
    
    try:
        from ipyfilechooser import FileChooser
        
        folder_chooser = FileChooser(
            path=str(Path.home()),
            title='Select your documents folder',
            show_only_dirs=True,
            select_default=True
        )
        print("📁 Select your documents folder below, then run Cell 2:")
        print("   (The picker is non-blocking - select BEFORE running the next cell)")
        display(folder_chooser)
        
    except ImportError:
        # Fallback: manual path entry
        print("Folder picker not available (ipyfilechooser not installed).")
        print(f"\nUsing default folder: {Path(DOC_FOLDER).absolute()}")
        print("\nTo use a different folder, edit DOC_FOLDER in this cell:")
        print("  DOC_FOLDER = '/path/to/your/documents'")
        os.makedirs(DOC_FOLDER, exist_ok=True)

Running locally

📁 Select your documents folder below, then run Cell 2:
   (The picker is non-blocking - select BEFORE running the next cell)


FileChooser(path='/mnt/sdb/arafat', filename='', title='Select your documents folder', show_hidden=False, sele…

In [134]:
# =============================================================================
# CELL 2: CONFIRM SELECTION AND LIST DOCUMENTS
# =============================================================================
# If you used a folder picker above, make sure you selected a folder
# BEFORE running this cell. The picker is non-blocking.
# =============================================================================

# Read selection from folder picker (if one was used)
if folder_chooser is not None and folder_chooser.selected_path:
    DOC_FOLDER = folder_chooser.selected_path
    print(f"✓ Using selected folder: {DOC_FOLDER}")
elif folder_chooser is not None:
    print("⚠ No folder selected in picker!")
    print("  Please go back to Cell 1, select a folder, then run this cell again.")
else:
    # No picker used (upload or manual path)
    print(f"✓ Using folder: {DOC_FOLDER}")

# Confirm folder (listing skipped for speed)
doc_path = Path(DOC_FOLDER)
if doc_path.exists():
    print(f"✓ Folder set: {doc_path.absolute()}")
    print("  Run the next cells to load, chunk, and index documents.")
else:
    print(f"⚠ Folder not found: {DOC_FOLDER}")
    print("  Please set DOC_FOLDER in the previous cell and run it again.")

✓ Using selected folder: /mnt/sdb/arafat/agentic_ai/Topic5RAG/Corpora/NewModelT
✓ Folder set: /mnt/sdb/arafat/agentic_ai/Topic5RAG/Corpora/NewModelT
  Run the next cells to load, chunk, and index documents.


---
## Stage 1: Document Loading

We need to extract text from our documents. For PDFs with embedded text,
PyMuPDF (fitz) reads the text layer directly - no OCR needed.

**Corpora:** Use PDFs from `Corpora/<name>/pdf_embedded/`. The `.txt` files in `txt/` are for checking retrieval vs OCR issues.

In [135]:
# Exercise 1 (and reuse): Official query lists. Reference: CR Jan 13, 20, 21, 23, 2026.
QUERIES_MODEL_T = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
]
QUERIES_CR = [
    "What did Mr. Flood have to say about Mayor David Black in Congress on January 13, 2026?",
    "What mistake did Elise Stefanik make in Congress on January 23, 2026?",
    "What is the purpose of the Main Street Parity Act?",
    "Who in Congress has spoken for and against funding of pregnancy centers?",
]

In [136]:
import fitz  # PyMuPDF
from typing import List, Tuple

def load_text_file(filepath: str) -> str:
    """Load a plain text file."""
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()


def load_pdf_file(filepath: str) -> str:
    """
    Extract text from a PDF with embedded text.
    
    PyMuPDF reads the text layer directly.
    For scanned PDFs without embedded text, you'd need OCR.
    """
    doc = fitz.open(filepath)
    text_parts = []
    
    for page_num, page in enumerate(doc):
        text = page.get_text()
        if text.strip():
            # Add page marker for debugging/citation
            text_parts.append(f"\n[Page {page_num + 1}]\n{text}")
    
    doc.close()
    return "\n".join(text_parts)


def load_documents(doc_folder: str) -> List[Tuple[str, str]]:
    """Load all documents from a folder. Returns list of (filename, content)."""
    documents = []
    folder = Path(doc_folder)
    
    for filepath in folder.rglob("*"):
        try:
            if not filepath.is_file():
                continue
        except OSError:
            continue
        if filepath.suffix.lower() not in ('.pdf', '.txt', '.md', '.text'):
            continue
        try:
            if filepath.suffix.lower() == '.pdf':
                content = load_pdf_file(str(filepath))
            elif filepath.suffix.lower() in ['.txt', '.md', '.text']:
                content = load_text_file(str(filepath))
            else:
                continue

            if content.strip():
                documents.append((filepath.name, content))
                print(f"✓ Loaded: {filepath.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ Error loading {filepath}: {e}")
    
    return documents

In [137]:
# Load your documents
documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")

if len(documents) == 0:
    print("\n⚠ No documents loaded! Please add PDF or TXT files to the documents folder.")

✓ Loaded: ModelTNew.txt (545,492 chars)
✓ Loaded: ModelTNew.pdf (469,891 chars)

Loaded 2 documents


In [138]:
# Inspect a document to verify loading worked
if documents:
    filename, content = documents[0]
    print(f"First document: {filename}")
    print(f"Total length: {len(content):,} characters")
    print(f"\nFirst 1000 characters:\n{'-'*40}")
    print(content[:1000])

First document: ModelTNew.txt
Total length: 545,492 characters

First 1000 characters:
----------------------------------------
SERVI

 Detailed Instructions for
  Servicing Ford Gars




    PRICE $250



         Published by




 DETROIT, MICHIGAN, U. S. A.
                                         Contents

Foreword . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .    111
Essentials of good service. . . . : . . . . . : . . . . . . . . . . . . . . . . . . . . . . .               ix
Ideal shop layout for average size dealer. . . . . . . . . . . . . . . . . . . . .                           x
Essential shop equipment. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .                  xi
The parts department. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
                                                                                                            ...
                                   

---
## Stage 2: Chunking

Documents need to be split into pieces small enough to be relevant but large enough to carry meaning.

**Why overlap?** If a key sentence sits right at a chunk boundary, splitting without overlap might cut it in half. Overlap ensures that information near boundaries appears intact in at least one chunk.

**Experiment:** Try different chunk sizes (256, 512, 1024) and see how it affects retrieval!

In [139]:
from dataclasses import dataclass

@dataclass
class Chunk:
    """A chunk of text with metadata for tracing back to source."""
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int


def chunk_text(
    text: str,
    source_file: str,
    chunk_size: int = 512,
    chunk_overlap: int = 128
) -> List[Chunk]:
    """
    Split text into overlapping chunks.
    
    We try to break at sentence or paragraph boundaries
    to avoid cutting mid-thought.
    """
    chunks = []
    start = 0
    chunk_index = 0
    
    while start < len(text):
        end = start + chunk_size
        
        # Try to break at a good boundary
        if end < len(text):
            # Look for paragraph break first
            para_break = text.rfind('\n\n', start + chunk_size // 2, end)
            if para_break != -1:
                end = para_break + 2
            else:
                # Look for sentence break
                sentence_break = text.rfind('. ', start + chunk_size // 2, end)
                if sentence_break != -1:
                    end = sentence_break + 2
        
        chunk_text_str = text[start:end].strip()
        
        if chunk_text_str:
            chunks.append(Chunk(
                text=chunk_text_str,
                source_file=source_file,
                chunk_index=chunk_index,
                start_char=start,
                end_char=end
            ))
            chunk_index += 1
        
        # Move forward, accounting for overlap
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end  # Safety: ensure progress
    
    return chunks

In [140]:
# ============================================
# EXPERIMENT: Try different chunk sizes!
# ============================================
CHUNK_SIZE = 256      # Try: 256, 512, 1024
CHUNK_OVERLAP = 128   # Try: 64, 128, 256
# For Ex 7/8 use rebuild_pipeline() — see cell after FAISS index.

# Chunk all documents
all_chunks = []
for filename, content in documents:
    doc_chunks = chunk_text(content, filename, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(doc_chunks)
    print(f"{filename}: {len(doc_chunks)} chunks")

print(f"\nTotal: {len(all_chunks)} chunks")

ModelTNew.txt: 5435 chunks
ModelTNew.pdf: 4870 chunks

Total: 10305 chunks


In [141]:
# Inspect some chunks
if all_chunks:
    print("Sample chunks:")
    indices_to_show = [0, len(all_chunks)//2, -1] if len(all_chunks) > 2 else range(len(all_chunks))
    for i in indices_to_show:
        chunk = all_chunks[i]
        print(f"\n{'='*60}")
        print(f"Chunk {chunk.chunk_index} from {chunk.source_file}")
        print(f"{'='*60}")
        print(chunk.text[:300] + "..." if len(chunk.text) > 300 else chunk.text)

Sample chunks:

Chunk 0 from ModelTNew.txt
SERVI

 Detailed Instructions for
  Servicing Ford Gars




    PRICE $250



         Published by




 DETROIT, MICHIGAN, U. S. A.
                                         Contents

Chunk 5152 from ModelTNew.txt
34-56-57
                              Note: N umbers r efe r t o p a ragra phs.
                                                  289
290                                   INDEX- Continued
                                                  B

Chunk 4869 from ModelTNew.pdf
ter (Page 287) · 
not equipped with starter (Page 288) 
trouble chart (Page 233) 
Note : Numbers refer to paragraphs.


---
## Stage 3: Embedding

Embeddings map text to dense vectors where **semantic similarity = geometric proximity**.

A sentence about "cardiac arrest" and one about "heart attack" will have similar embeddings even though they share no words.

**Note:** sentence-transformers does NOT auto-detect Apple MPS - we must pass the device explicitly.

In [142]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
# Options:
# - "sentence-transformers/all-MiniLM-L6-v2": Fast, small (80MB), good quality
# - "BAAI/bge-small-en-v1.5": Better for retrieval, similar size

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBEDDING_MODEL}")
print(f"Device: {DEVICE}")

# Must explicitly pass device for MPS support!
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {EMBEDDING_DIM}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


In [143]:
# DEMO: See how embeddings capture semantic similarity
test_sentences = [
    "The engine needs regular oil changes.",
    "Motor oil should be replaced periodically.",
    "The Senate convened at noon.",
    "Congress began its session at midday."
]

test_embeddings = embed_model.encode(test_sentences)

# Compute cosine similarity matrix
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print("Cosine similarity matrix:")
print("\n" + " " * 40 + "  [0]    [1]    [2]    [3]")
for i, s1 in enumerate(test_sentences):
    sims = [cosine_sim(test_embeddings[i], test_embeddings[j]) for j in range(4)]
    print(f"[{i}] {s1[:35]:35} {sims[0]:.3f}  {sims[1]:.3f}  {sims[2]:.3f}  {sims[3]:.3f}")

print("\n→ Notice: [0]-[1] are similar (both about oil), [2]-[3] are similar (both about Congress)")

Cosine similarity matrix:

                                          [0]    [1]    [2]    [3]
[0] The engine needs regular oil change 1.000  0.728  -0.045  -0.032
[1] Motor oil should be replaced period 0.728  1.000  0.014  0.035
[2] The Senate convened at noon.        -0.045  0.014  1.000  0.684
[3] Congress began its session at midda -0.032  0.035  0.684  1.000

→ Notice: [0]-[1] are similar (both about oil), [2]-[3] are similar (both about Congress)


In [144]:
# Embed all chunks - this may take a few minutes for large corpora
if all_chunks:
    print(f"Embedding {len(all_chunks)} chunks on {DEVICE}...")
    chunk_texts = [c.text for c in all_chunks]
    chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)
    chunk_embeddings = chunk_embeddings.astype('float32')  # FAISS wants float32
    print(f"Embeddings shape: {chunk_embeddings.shape}")
else:
    print("No chunks to embed - please load documents first.")

Embedding 10305 chunks on cuda...


Batches:   0%|          | 0/323 [00:00<?, ?it/s]

Embeddings shape: (10305, 384)


---
## Stage 4: Vector Index (FAISS)

FAISS efficiently finds nearest neighbors in high-dimensional spaces.

We use a simple **flat index** (brute-force search) which is transparent and works well for up to ~100k vectors. For larger corpora, you'd use approximate methods like IVF or HNSW.

**Note:** FAISS GPU support is CUDA-only. On MPS/CPU, we use faiss-cpu (still very fast for <100k vectors).

In [145]:
import faiss

# Create FAISS index
# IndexFlatIP = Inner Product (for cosine similarity on normalized vectors)
index = faiss.IndexFlatIP(EMBEDDING_DIM)

if all_chunks:
    # Normalize vectors so inner product = cosine similarity
    faiss.normalize_L2(chunk_embeddings)
    
    # Add vectors to index
    index.add(chunk_embeddings)
    print(f"Index built with {index.ntotal} vectors")
else:
    print("No embeddings to index - please load and embed documents first.")

Index built with 10305 vectors


---
## Stage 5: Retrieval

Now we can search! Given a query, we:
1. Embed the query with the same model
2. Find the top-k most similar chunks
3. Return those chunks as context

In [146]:
# Helper for Exercises 7 & 8: rebuild chunks + index with different chunk_size / chunk_overlap.
def rebuild_pipeline(chunk_size: int = 512, chunk_overlap: int = 128):
    """Re-chunk documents, re-embed, and rebuild FAISS index. Updates global all_chunks and index."""
    global all_chunks, index
    all_chunks = []
    for filename, content in documents:
        all_chunks.extend(chunk_text(content, filename, chunk_size=chunk_size, chunk_overlap=chunk_overlap))
    chunk_embeddings = embed_model.encode([c.text for c in all_chunks], show_progress_bar=True).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks, chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")

In [147]:
def retrieve(query: str, top_k: int = 5):
    """
    Retrieve the top-k most relevant chunks for a query.
    
    Returns: List of (chunk, similarity_score) tuples
    """
    # Embed the query
    query_embedding = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)
    
    # Search
    scores, indices = index.search(query_embedding, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append((all_chunks[idx], float(score)))
    
    return results

In [148]:
# Test retrieval
# ============================================
# TRY DIFFERENT QUERIES FOR YOUR CORPUS!
# ============================================
test_query = "What is the procedure for engine maintenance?"  # ← Modify this!

if index.ntotal > 0:
    results = retrieve(test_query, top_k=5)
    
    print(f"Query: {test_query}\n")
    print("Top 5 retrieved chunks:")
    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n[{i}] Score: {score:.4f} | Source: {chunk.source_file}")
        print(f"    {chunk.text[:200]}...")
else:
    print("Index is empty - please load, chunk, and embed documents first.")

Query: What is the procedure for engine maintenance?

Top 5 retrieved chunks:

[1] Score: 0.5602 | Source: ModelTNew.txt
    assembling, clean all parts thoroughly, also lubricate all
  moving parts and the surfaces upon which they move, such as
  bearings, bushings, pistons, cylinders, etc....

[2] Score: 0.5509 | Source: ModelTNew.pdf
    . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1 
55 
11 
Assemble valves and springs, and check timing of engine . . 
20 
12 
Overhaul transmission, including checking and replacing 
magnets...

[3] Score: 0.5500 | Source: ModelTNew.pdf
    cleaning. After the cleaning operation it is trans- 
ferred to the stand or repair bench on which the work is to be performed. 
When the overhaul work is completed the assembly is returned by 
means o...

[4] Score: 0.5431 | Source: ModelTNew.pdf
    en a car is brought in for major repair 
work is to first assign the car to a section of the shop set aside for re- 
pair jobs. The assembly to be overhaul

---
## Stage 6: Generation (LLM)

Now we load a local LLM to generate answers from the retrieved context.

**Recommended models:**
- `Qwen/Qwen2.5-1.5B-Instruct` - Best instruction following at this size
- `Qwen/Qwen2.5-3B-Instruct` - Even better if you have 8GB+ VRAM
- `meta-llama/Llama-3.2-1B-Instruct` - Alternative, slightly weaker

**Device handling:**
- CUDA: Uses `device_map="auto"` and float16
- MPS: Loads to CPU first, then moves to MPS with float32
- CPU: Uses float32 (slower but works)

In [149]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# ============================================
# CHOOSE YOUR MODEL
# ============================================
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # Or try "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading LLM: {LLM_MODEL}")
print(f"Device: {DEVICE}, Dtype: {DTYPE}")
print("This may take a few minutes on first run...\n")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

# Load with appropriate settings for each device type
if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        device_map="auto",
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    print("Model loaded on CUDA")
    
elif DEVICE == 'mps':
    # For MPS, load to CPU first, then move to MPS
    # (device_map="auto" doesn't work well with MPS)
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    model = model.to(DEVICE)
    print("Model loaded on MPS (Apple Silicon)")
    
else:
    # CPU
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=DTYPE,
        trust_remote_code=True
    )
    print("Model loaded on CPU (this will be slow)")

Loading LLM: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda, Dtype: torch.float16
This may take a few minutes on first run...



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded on CUDA


In [150]:
def generate_response(prompt: str, max_new_tokens: int = 512, temperature: float = 0.3) -> str:
    """
    Generate a response from the LLM.
    
    Lower temperature = more focused/deterministic
    Higher temperature = more creative/random
    """
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Move inputs to the correct device
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the new tokens
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return response.strip()

---
## Stage 7: The Complete RAG Pipeline

Now we put it all together. The **prompt template** is critical - it must instruct the model to use the retrieved context.

In [151]:
# The RAG prompt template
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer the question based ONLY on the information in the context above
- If the context doesn't contain enough information to answer, say so
- Quote relevant parts of the context to support your answer
- Be concise and direct

ANSWER:"""


def direct_query(question: str, max_new_tokens: int = 512) -> str:
    """Ask the LLM directly with no retrieved context (for RAG vs no-RAG comparison)."""
    prompt = f"""Answer this question:
{question}

Answer:"""
    return generate_response(prompt, max_new_tokens=max_new_tokens)

def rag_query(question: str, top_k: int = 5, show_context: bool = False, prompt_template: str = None) -> str:
    """The complete RAG pipeline. prompt_template: custom template for Exercise 10."""
    # Step 1: Retrieve
    results = retrieve(question, top_k)
    
    # Format context
    context_parts = []
    for chunk, score in results:
        context_parts.append(f"[Source: {chunk.source_file}, Relevance: {score:.3f}]\n{chunk.text}")
    context = "\n\n---\n\n".join(context_parts)
    
    if show_context:
        print("=" * 60)
        print("RETRIEVED CONTEXT:")
        print("=" * 60)
        print(context)
        print("=" * 60 + "\n")
    
    # Step 2: Build prompt (use custom template if provided)
    template = prompt_template if prompt_template is not None else PROMPT_TEMPLATE
    prompt = template.format(context=context, question=question)
    
    # Step 3: Generate
    answer = generate_response(prompt)
    
    return answer

In [152]:
# ============================================
# TEST YOUR RAG PIPELINE!
# ============================================

question = "What maintenance is required for the engine?"  # ← Modify for your corpus!

if index.ntotal > 0:
    print(f"Question: {question}\n")
    print("Generating answer...\n")
    
    answer = rag_query(question, top_k=5, show_context=True)
    
    print("ANSWER:")
    print(answer)
else:
    print("Pipeline not ready - please complete all previous stages first.")

Question: What maintenance is required for the engine?

Generating answer...

RETRIEVED CONTEXT:
[Source: ModelTNew.pdf, Relevance: 0.601]
. . . . . . . . . . . . . . . . . . . . . . . . . . . . 1 
55 
11 
Assemble valves and springs, and check timing of engine . . 
20 
12 
Overhaul transmission, including checking and replacing 
magnets . . . . . . . . . . . . . . . . . . . . . . . . . . .

---

[Source: ModelTNew.txt, Relevance: 0.596]
. . . . . . . . . . . . . . . . . . . . . . . . . . . 1    55
11    Assemble valves and springs, and check timing of engine . .                                      20
12    Overhaul transmission, including checking and replacing
        magnets . . . . .

---

[Source: ModelTNew.txt, Relevance: 0.568]
assembling, clean all parts thoroughly, also lubricate all
  moving parts and the surfaces upon which they move, such as
  bearings, bushings, pistons, cylinders, etc.

---

[Source: ModelTNew.txt, Relevance: 0.553]
. . . . . . . . . . . . . . . . . . . 

---
## Exercise 1: RAG vs No-RAG Comparison (Course Assignment)

Run the cells below to execute Exercise 1 and save outputs to `outputs/exercise1/`.

**Part A:** Model T corpus (uses current index if you loaded NewModelT)  
**Part B:** Congressional Record corpus (reloads, re-chunks, re-embeds)  
**Part C:** Observations summary

In [121]:
# =============================================================================
# EXERCISE 1 PART A: Model T queries (no RAG + RAG)
# =============================================================================
# Run this cell when the index is built from Corpora/NewModelT.
# Saves to outputs/exercise1/model_t_results.txt
# =============================================================================

import os
OUTPUT_DIR = "outputs/exercise1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def run_exercise1_queries(queries, corpus_name, outpath):
    """Run each query with direct_query and rag_query, save results."""
    lines = [f"# Exercise 1: {corpus_name}", f"# Corpus: {DOC_FOLDER}", ""]
    for i, q in enumerate(queries, 1):
        lines.append(f"{'='*70}")
        lines.append(f"QUERY {i}: {q}")
        lines.append(f"{'='*70}")
        lines.append("")
        lines.append("NO RAG:")
        lines.append("-" * 40)
        try:
            no_rag = direct_query(q)
            lines.append(no_rag)
        except Exception as e:
            lines.append(f"[Error: {e}]")
        lines.append("")
        lines.append("RAG (top_k=5):")
        lines.append("-" * 40)
        try:
            results = retrieve(q, top_k=5)
            for j, (chunk, score) in enumerate(results, 1):
                lines.append(f"  [{j}] Score: {score:.4f} | {chunk.source_file}")
                lines.append(f"      {chunk.text[:150]}...")
            lines.append("")
            rag_ans = rag_query(q, top_k=5)
            lines.append("Answer:")
            lines.append(rag_ans)
        except Exception as e:
            lines.append(f"[Error: {e}]")
        lines.append("")
    with open(outpath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"✓ Saved to {outpath}")

if index.ntotal > 0:
    print("Running Model T queries...")
    run_exercise1_queries(QUERIES_MODEL_T, "Model T Ford Manual", f"{OUTPUT_DIR}/model_t_results.txt")
else:
    print("Index empty. Load Model T corpus first (DOC_FOLDER = Corpora/NewModelT), then run load/chunk/embed/index cells.")

Running Model T queries...
✓ Saved to outputs/exercise1/model_t_results.txt


In [122]:
# =============================================================================
# EXERCISE 1 PART B: Switch to Congressional Record and rebuild index
# =============================================================================
# Run this cell to load CR corpus, re-chunk, re-embed, rebuild FAISS index.
# Then run the next cell for CR queries.
# =============================================================================

CR_FOLDER = "Corpora/Congressional_Record_Jan_2026/txt"
if Path(CR_FOLDER).exists():
    DOC_FOLDER = CR_FOLDER
    print(f"Loading Congressional Record from: {DOC_FOLDER}")
    documents = load_documents(DOC_FOLDER)
    print(f"Loaded {len(documents)} documents")
    
    # Re-chunk
    all_chunks = []
    for filename, content in documents:
        doc_chunks = chunk_text(content, filename, CHUNK_SIZE, CHUNK_OVERLAP)
        all_chunks.extend(doc_chunks)
    print(f"Total chunks: {len(all_chunks)}")
    
    # Re-embed
    chunk_texts = [c.text for c in all_chunks]
    chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True).astype("float32")
    
    # Rebuild FAISS index
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"✓ Index rebuilt with {index.ntotal} vectors (Congressional Record)")
else:
    print(f"⚠ Folder not found: {CR_FOLDER}")
    print("  Unzip Corpora.zip and ensure Congressional_Record_Jan_2026/txt exists.")

Loading Congressional Record from: Corpora/Congressional_Record_Jan_2026/txt
✓ Loaded: CREC-2026-01-02.txt (17,600 chars)
✓ Loaded: CREC-2026-01-12.txt (624,304 chars)
✓ Loaded: CREC-2026-01-07.txt (681,194 chars)
✓ Loaded: CREC-2026-01-08.txt (1,304,929 chars)
✓ Loaded: CREC-2026-01-08-bk3.txt (1,150,111 chars)
✓ Loaded: CREC-2026-01-29.txt (416,102 chars)
✓ Loaded: CREC-2026-01-26-bk2.txt (12,984 chars)
✓ Loaded: CREC-2026-01-20.txt (1,091,304 chars)
✓ Loaded: CREC-2026-01-21.txt (447,471 chars)
✓ Loaded: CREC-2026-01-13.txt (950,188 chars)
✓ Loaded: CREC-2026-01-14.txt (1,686,048 chars)
✓ Loaded: CREC-2026-01-09.txt (225,967 chars)
✓ Loaded: CREC-2026-01-28.txt (507,125 chars)
✓ Loaded: CREC-2026-01-15.txt (603,144 chars)
✓ Loaded: CREC-2026-01-22.txt (1,823,848 chars)
✓ Loaded: CREC-2026-01-27.txt (233,979 chars)
✓ Loaded: CREC-2026-01-30.txt (334,117 chars)
✓ Loaded: CREC-2026-01-08-bk2.txt (27,224 chars)
✓ Loaded: CREC-2026-01-22-bk2.txt (1,611,370 chars)
✓ Loaded: CREC-2026-01-2

Batches:   0%|          | 0/4387 [00:00<?, ?it/s]

✓ Index rebuilt with 140359 vectors (Congressional Record)


In [123]:
# =============================================================================
# EXERCISE 1 PART B (continued): Congressional Record queries
# =============================================================================
# Run this cell AFTER the previous cell (CR index must be built).
# Saves to outputs/exercise1/congressional_record_results.txt
# =============================================================================

if index.ntotal > 0:
    print("Running Congressional Record queries...")
    run_exercise1_queries(QUERIES_CR, "Congressional Record Jan 2026", f"{OUTPUT_DIR}/congressional_record_results.txt")
else:
    print("Index empty. Run the previous cell first to load and index Congressional Record.")

Running Congressional Record queries...
✓ Saved to outputs/exercise1/congressional_record_results.txt


In [124]:
# =============================================================================
# EXERCISE 1 PART C: Create observations template
# =============================================================================
# Creates outputs/exercise1/observations.md for you to fill in.
# Document: hallucination without RAG? RAG grounding? General knowledge correct?
# =============================================================================

obs = """# Exercise 1 Observations

## Model T Queries
| Query | Hallucinates (no RAG)? | RAG grounds answer? | General knowledge correct? |
|-------|------------------------|---------------------|----------------------------|
| Carburetor adjustment | | | |
| Spark plug gap | | | |
| Slipping transmission band | | | |
| Engine oil | | | |

## Congressional Record Queries
| Query | Hallucinates (no RAG)? | RAG grounds answer? | General knowledge correct? |
|-------|------------------------|---------------------|----------------------------|
| Mr. Flood / Mayor Black (Jan 13) | | | |
| Elise Stefanik mistake (Jan 23) | | | |
| Main Street Parity Act (Jan 20) | | | |
| Pregnancy centers funding (Jan 21) | | | |

## Summary
- [Add notes on when RAG helped vs. when model's knowledge sufficed]
- [Note any hallucinations and whether RAG corrected them]
"""
with open(f"{OUTPUT_DIR}/observations.md", "w", encoding="utf-8") as f:
    f.write(obs)
print(f"✓ Created {OUTPUT_DIR}/observations.md — fill in the table after reviewing results.")

✓ Created outputs/exercise1/observations.txt — fill in the table after reviewing results.


---
## Exercise 2: GPT-4o Mini (No RAG) — Large Model Comparison

Run the cell below to query GPT-4o Mini on the same 8 queries (no retrieval).
Requires: `pip install openai` and `OPENAI_API_KEY` environment variable.

**Alternative:** Run the standalone script from terminal: `python run_exercise2.py`

In [131]:
# =============================================================================
# EXERCISE 2: Query GPT-4o Mini (no RAG) on same queries as Exercise 1
# =============================================================================
# Requires: pip install openai, export OPENAI_API_KEY='...'
# Saves to outputs/exercise2/gpt4o_mini_results.md
# =============================================================================

import os
EX2_OUTPUT = "outputs/exercise2"
os.makedirs(EX2_OUTPUT, exist_ok=True)

def query_gpt4o_mini(question: str, model: str = "gpt-4o-mini") -> str:
    from openai import OpenAI
    client = OpenAI()
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}],
        max_tokens=512,
        temperature=0.3,
    )
    return r.choices[0].message.content.strip()

if os.environ.get("OPENAI_API_KEY"):
    lines = ["# Exercise 2: GPT-4o Mini (No RAG)\n"]
    for name, queries in [("Model T", QUERIES_MODEL_T), ("Congressional Record", QUERIES_CR)]:
        lines.append(f"## {name}\n")
        for i, q in enumerate(queries, 1):
            lines.append(f"### Q{i}: {q}\n")
            try:
                ans = query_gpt4o_mini(q)
                lines.append(ans + "\n")
            except Exception as e:
                lines.append(f"[Error: {e}]\n")
            lines.append("---\n")
    with open(f"{EX2_OUTPUT}/gpt4o_mini_results.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"✓ Saved to {EX2_OUTPUT}/gpt4o_mini_results.md")
    obs = """# Exercise 2 Observations\n\nCompare to outputs/exercise1 (Qwen no-RAG). Fill in: which questions did GPT-4o Mini get right? Better hallucination avoidance? Training cutoff vs Jan 2026 CR?\n"""
    with open(f"{EX2_OUTPUT}/observations.md", "w", encoding="utf-8") as f:
        f.write(obs)
    print(f"✓ Created {EX2_OUTPUT}/observations.md")
else:
    print("Set OPENAI_API_KEY. Or run from terminal: python run_exercise2.py")

Set OPENAI_API_KEY. Or run from terminal: python run_exercise2.py


---
## Exercise 3: Local RAG vs Frontier Chat Model

**Manual workflow:** Compare local Qwen+RAG (Model T) vs GPT-4/Claude via web (no file upload).

1. Local RAG answers: see `outputs/exercise1/model_t_results.txt` (RAG sections)
2. Go to chat.openai.com or claude.ai, ask the 4 Model T queries (no file upload)
3. Paste frontier answers into `outputs/exercise3/frontier_answers_template.md`
4. Fill in `outputs/exercise3/observations.md`

See `outputs/exercise3/RUN_INSTRUCTIONS.md` for full steps.

---
## Exercise 4: Effect of Top-K Retrieval Count

Vary k = 1, 3, 5, 10, 20. Run same queries, record answer quality and latency.
Saves to `outputs/exercise4/`. Ensure Model T (or desired corpus) is loaded.

In [153]:
# =============================================================================
# EXERCISE 4: Effect of top-K (k = 1, 3, 5, 10, 20)
# =============================================================================
# Runs 4 Model T queries for each k, records latency, saves to outputs/exercise4/
# =============================================================================

import os
import time
EX4_OUTPUT = "outputs/exercise4"
os.makedirs(EX4_OUTPUT, exist_ok=True)

# Use first 4 Model T queries (or change to QUERIES_CR for Congressional Record)
EX4_QUERIES = QUERIES_MODEL_T[:4]
K_VALUES = [1, 3, 5, 10, 20]

results_by_k = {}
if index.ntotal > 0:
    for k in K_VALUES:
        results_by_k[k] = []
        print(f"\n--- top_k = {k} ---")
        for i, q in enumerate(EX4_QUERIES):
            t0 = time.time()
            ans = rag_query(q, top_k=k)
            elapsed = time.time() - t0
            results_by_k[k].append({"query": q, "answer": ans, "latency_sec": elapsed})
            print(f"  Q{i+1}: {elapsed:.2f}s")
    
    # Save results
    lines = ["# Exercise 4: Effect of Top-K", f"# Corpus: {DOC_FOLDER}", ""]
    for k in K_VALUES:
        lines.append(f"\n## top_k = {k}\n")
        for i, r in enumerate(results_by_k[k], 1):
            lines.append(f"### Query {i}: {r['query']}")
            lines.append(f"Latency: {r['latency_sec']:.2f}s")
            lines.append("")
            lines.append(r['answer'][:800] + ("..." if len(r['answer']) > 800 else ""))
            lines.append("\n---\n")
    with open(f"{EX4_OUTPUT}/topk_results.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"\n✓ Saved to {EX4_OUTPUT}/topk_results.md")
    
    # Latency summary
    lat_lines = ["# Top-K Latency Summary (seconds)\n", "| k | Q1 | Q2 | Q3 | Q4 | Avg |"]
    lat_lines.append("|---|----|----|----|----|-----|")
    for k in K_VALUES:
        lats = [r["latency_sec"] for r in results_by_k[k]]
        avg = sum(lats) / len(lats)
        lat_lines.append(f"| {k} | " + " | ".join(f"{x:.2f}" for x in lats) + f" | {avg:.2f} |")
    with open(f"{EX4_OUTPUT}/latency_summary.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lat_lines))
    print(f"✓ Saved to {EX4_OUTPUT}/latency_summary.md")
else:
    print("Index empty. Load corpus and run pipeline first.")


--- top_k = 1 ---
  Q1: 6.10s
  Q2: 5.27s
  Q3: 3.49s
  Q4: 3.88s

--- top_k = 3 ---
  Q1: 21.54s
  Q2: 5.01s
  Q3: 24.15s
  Q4: 5.18s

--- top_k = 5 ---
  Q1: 8.62s
  Q2: 9.71s
  Q3: 4.66s
  Q4: 3.26s

--- top_k = 10 ---
  Q1: 10.51s
  Q2: 8.19s
  Q3: 7.55s
  Q4: 2.71s

--- top_k = 20 ---
  Q1: 10.43s
  Q2: 6.77s
  Q3: 10.79s
  Q4: 6.53s

✓ Saved to outputs/exercise4/topk_results.md
✓ Saved to outputs/exercise4/latency_summary.md


In [154]:
# Create observations template for Exercise 4
obs = """# Exercise 4 Observations

## Answer Quality by top_k

| k | Carburetor | Spark plug | Transmission | Engine oil | Notes |
|---|------------|------------|--------------|------------|-------|
| 1 | | | | | |
| 3 | | | | | |
| 5 | | | | | |
| 10 | | | | | |
| 20 | | | | | |

## Latency

See latency_summary.md. Does latency increase with k?

## When does adding context stop helping?

[At what k did quality plateau or decline?]

## When does too much context hurt?

[Did k=10 or k=20 introduce irrelevant info or confusion?]

## Summary

- Optimal k for this corpus: ___
- Trade-off: more context vs. latency, noise
"""
with open(f"{EX4_OUTPUT}/observations.md", "w", encoding="utf-8") as f:
    f.write(obs)
print(f"✓ Created {EX4_OUTPUT}/observations.md — fill in after reviewing results.")

✓ Created outputs/exercise4/observations.md — fill in after reviewing results.


---
## Exercise 5: Handling Unanswerable Questions

Test: off-topic, related-but-not-in-corpus, false-premise. With and without strict prompt.
Saves to `outputs/exercise5/`. Use any corpus (Model T, CR, etc.).

In [155]:
# =============================================================================
# EXERCISE 5: Unanswerable questions (default prompt vs strict prompt)
# =============================================================================
# Types: off-topic, related-not-in-corpus, false-premise
# Saves to outputs/exercise5/
# =============================================================================

import os
EX5_OUTPUT = "outputs/exercise5"
os.makedirs(EX5_OUTPUT, exist_ok=True)

# Strict prompt: encourages "I cannot answer"
PROMPT_STRICT = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer the question based ONLY on the information in the context above
- If the context doesn't contain the answer, say "I cannot answer this from the available documents."
- Do NOT use your general knowledge. Do NOT guess or make up information.
- If you can answer from context, quote relevant parts to support your answer.

ANSWER:"""

EX5_QUERIES = [
    ("off_topic", "What is the capital of France?"),
    ("related_not_in_corpus", "What's the horsepower of a 1925 Model T?"),
    ("false_premise", "Why does the manual recommend synthetic oil?"),
]

if index.ntotal > 0:
    lines = ["# Exercise 5: Unanswerable Questions", f"# Corpus: {DOC_FOLDER}", ""]
    for qtype, q in EX5_QUERIES:
        lines.append(f"\n## {qtype}: {q}\n")
        lines.append("### Default prompt\n")
        ans_default = rag_query(q, top_k=5, show_context=False)
        lines.append(ans_default)
        lines.append("\n### Strict prompt (I cannot answer...)\n")
        ans_strict = rag_query(q, top_k=5, show_context=False, prompt_template=PROMPT_STRICT)
        lines.append(ans_strict)
        lines.append("\n---\n")
    with open(f"{EX5_OUTPUT}/unanswerable_results.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"✓ Saved to {EX5_OUTPUT}/unanswerable_results.md")
else:
    print("Index empty. Load corpus and run pipeline first.")

✓ Saved to outputs/exercise5/unanswerable_results.md


In [156]:
# Create observations template for Exercise 5
obs = """# Exercise 5 Observations

## Does the model admit it doesn't know?

| Question type | Default prompt | Strict prompt |
|---------------|----------------|---------------|
| Off-topic (capital of France) | | |
| Related not in corpus (1925 hp) | | |
| False premise (synthetic oil) | | |

## Does it hallucinate plausible-sounding but wrong answers?

[Note any hallucinations for each question type]

## Does retrieved context help or hurt?

[Does irrelevant context encourage hallucination?]

## Does the strict prompt help?

[Compare default vs strict — more "I cannot answer" with strict?]

## Summary

- [Key takeaways]
"""
with open("outputs/exercise5/observations.md", "w", encoding="utf-8") as f:
    f.write(obs)
print("✓ Created outputs/exercise5/observations.md — fill in after reviewing results.")

✓ Created outputs/exercise5/observations.md — fill in after reviewing results.


---
## Exercise 6: Query Phrasing Sensitivity

Same underlying question, 5+ phrasings. Compare top-5 retrieved chunks, scores, overlap.
Saves to `outputs/exercise6/`. Use any corpus (Model T, Learjet, etc.).

In [157]:
# =============================================================================
# EXERCISE 6: Query phrasing sensitivity
# =============================================================================
# Same question, 5+ phrasings. Record top-5 chunks, scores, overlap.
# Saves to outputs/exercise6/
# =============================================================================

import os
EX6_OUTPUT = "outputs/exercise6"
os.makedirs(EX6_OUTPUT, exist_ok=True)

# Underlying question: engine maintenance. 5+ phrasings (course examples + variants)
EX6_PHRASINGS = [
    ("formal", "What is the recommended maintenance schedule for the engine?"),
    ("casual", "How often should I service the engine?"),
    ("keywords", "engine maintenance intervals"),
    ("question_form", "When do I need to check the engine?"),
    ("indirect", "Preventive maintenance requirements"),
]

def chunk_id(chunk):
    return (chunk.source_file, chunk.chunk_index)

if index.ntotal > 0:
    results = {}
    for label, phrase in EX6_PHRASINGS:
        r = retrieve(phrase, top_k=5)
        results[label] = r
        print(f"{label}: {phrase[:50]}...")
    
    # Save detailed results
    lines = ["# Exercise 6: Query Phrasing Sensitivity", f"# Corpus: {DOC_FOLDER}", ""]
    for label, phrase in EX6_PHRASINGS:
        lines.append(f"\n## {label}: \"{phrase}\"\n")
        for i, (chunk, score) in enumerate(results[label], 1):
            lines.append(f"  [{i}] Score: {score:.4f} | {chunk.source_file}")
            lines.append(f"      {chunk.text[:120]}...")
        lines.append("")
    
    # Overlap analysis
    lines.append("\n## Overlap Analysis\n")
    chunk_sets = {label: set(chunk_id(c) for c, _ in results[label]) for label, _ in EX6_PHRASINGS}
    for i, (l1, _) in enumerate(EX6_PHRASINGS):
        for l2, _ in EX6_PHRASINGS[i+1:]:
            overlap = len(chunk_sets[l1] & chunk_sets[l2])
            lines.append(f"- {l1} ∩ {l2}: {overlap}/5 chunks overlap")
    
    with open(f"{EX6_OUTPUT}/phrasing_results.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"\n✓ Saved to {EX6_OUTPUT}/phrasing_results.md")
else:
    print("Index empty. Load corpus and run pipeline first.")

formal: What is the recommended maintenance schedule for t...
casual: How often should I service the engine?...
keywords: engine maintenance intervals...
question_form: When do I need to check the engine?...
indirect: Preventive maintenance requirements...

✓ Saved to outputs/exercise6/phrasing_results.md


In [158]:
# Create observations template for Exercise 6
obs = """# Exercise 6 Observations

## Top-5 retrieved chunks by phrasing

See phrasing_results.md for full output.

## Similarity scores (top chunk)

| Phrasing | Top score | Notes |
|----------|-----------|-------|
| formal | | |
| casual | | |
| keywords | | |
| question_form | | |
| indirect | | |

## Overlap between result sets

[Which phrasings retrieved similar chunks? See overlap section in phrasing_results.md]

## Which phrasing retrieved the best chunks?

[Compare relevance of top chunks for each phrasing]

## Do keyword-style queries work better or worse than natural questions?

[Your observation]

## Implications for query rewriting strategies

[What does this tell you about rewriting user queries before retrieval?]

## Summary

- [Key takeaways]
"""
with open("outputs/exercise6/observations.md", "w", encoding="utf-8") as f:
    f.write(obs)
print("✓ Created outputs/exercise6/observations.md — fill in after reviewing results.")

✓ Created outputs/exercise6/observations.md — fill in after reviewing results.


---
## Exercise 7: Chunk Overlap Experiment

Re-chunk with overlap 0, 64, 128, 256 (chunk_size=512 constant). Test a question whose answer may span chunk boundaries.

**Note:** Takes a long time (each rebuild re-embeds). Recommended on Colab with T4+ GPU.

In [159]:
# =============================================================================
# EXERCISE 7: Chunk overlap (0, 64, 128, 256) — chunk_size=512 constant
# =============================================================================
# Query whose answer may span chunk boundaries. Saves to outputs/exercise7/
# WARNING: Each rebuild re-embeds — takes ~several minutes per config
# =============================================================================

import os
EX7_OUTPUT = "outputs/exercise7"
os.makedirs(EX7_OUTPUT, exist_ok=True)

EX7_CHUNK_SIZE = 512
EX7_OVERLAPS = [0, 64, 128, 256]
# Question whose answer may span boundaries (procedure, multi-step)
EX7_QUERY = "What is the complete procedure for draining and replacing the oil in the crankcase?"

results = []
for overlap in EX7_OVERLAPS:
    print(f"\n--- Overlap = {overlap} ---")
    rebuild_pipeline(chunk_size=EX7_CHUNK_SIZE, chunk_overlap=overlap)
    n_chunks = len(all_chunks)
    retrieved = retrieve(EX7_QUERY, top_k=5)
    answer = rag_query(EX7_QUERY, top_k=5)
    results.append({
        "overlap": overlap,
        "n_chunks": n_chunks,
        "retrieved": retrieved,
        "answer": answer,
    })

# Save
lines = ["# Exercise 7: Chunk Overlap", f"# Corpus: {DOC_FOLDER}", f"# chunk_size={EX7_CHUNK_SIZE}", ""]
lines.append(f"Query: {EX7_QUERY}\n")
for r in results:
    lines.append(f"\n## Overlap = {r['overlap']} (n_chunks={r['n_chunks']})")
    lines.append("")
    for i, (chunk, score) in enumerate(r["retrieved"], 1):
        lines.append(f"  [{i}] Score: {score:.4f} | {chunk.source_file}")
        lines.append(f"      {chunk.text[:120]}...")
    lines.append("\nAnswer:")
    lines.append(r["answer"][:600] + ("..." if len(r["answer"]) > 600 else ""))
    lines.append("\n---")
with open(f"{EX7_OUTPUT}/overlap_results.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print(f"\n✓ Saved to {EX7_OUTPUT}/overlap_results.md")


--- Overlap = 0 ---


Batches:   0%|          | 0/73 [00:00<?, ?it/s]

Rebuilt: 2320 chunks, chunk_size=512, chunk_overlap=0

--- Overlap = 64 ---


Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Rebuilt: 2716 chunks, chunk_size=512, chunk_overlap=64

--- Overlap = 128 ---


Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Rebuilt: 3277 chunks, chunk_size=512, chunk_overlap=128

--- Overlap = 256 ---


Batches:   0%|          | 0/163 [00:00<?, ?it/s]

Rebuilt: 5215 chunks, chunk_size=512, chunk_overlap=256

✓ Saved to outputs/exercise7/overlap_results.md


In [160]:
# Create observations template for Exercise 7
obs = """# Exercise 7 Observations

## Top-5 retrieved chunks by overlap

See overlap_results.md for full output.

## Index size (n_chunks) by overlap

| Overlap | n_chunks | Notes |
|---------|----------|-------|
| 0 | | |
| 64 | | |
| 128 | | |
| 256 | | |

## Similarity scores (top chunk)

| Overlap | Top score | Notes |
|---------|-----------|-------|
| 0 | | |
| 64 | | |
| 128 | | |
| 256 | | |

## Does higher overlap improve retrieval of complete information?

[Compare answer completeness across overlap values. Does the oil-drain procedure span boundaries?]

## Cost: Index size, redundant information

[More chunks with higher overlap? Redundant context in retrieved chunks?]

## Point of diminishing returns?

[At what overlap did quality plateau?]

## Implications for chunking strategy

[What does this tell you about choosing overlap for boundary-spanning answers?]

## Summary

- [Key takeaways]
"""
with open("outputs/exercise7/observations.md", "w", encoding="utf-8") as f:
    f.write(obs)
print("✓ Created outputs/exercise7/observations.md — fill in after reviewing results.")

✓ Created outputs/exercise7/observations.md — fill in after reviewing results.


---
## Experiments: Understanding RAG Behavior

Now that you have a working pipeline, try these experiments to understand how each component affects the results.

---
## Exercise 8: Chunk Size Experiment

Re-chunk with sizes 128, 256, 512, 1024, 2048 (overlap scales with size). Test retrieval quality and answer completeness.

**Note:** Takes a long time (each rebuild re-embeds). Recommended on Colab with T4+ GPU.

In [161]:
# =============================================================================
# EXERCISE 8: Chunk size (128, 256, 512, 1024, 2048) — overlap scales
# =============================================================================
# Query: oil drain procedure (same as Ex 7). Saves to outputs/exercise8/
# WARNING: Each rebuild re-embeds — takes ~several minutes per config
# =============================================================================

import os
EX8_OUTPUT = "outputs/exercise8"
os.makedirs(EX8_OUTPUT, exist_ok=True)

EX8_CHUNK_SIZES = [128, 256, 512, 1024, 2048]
# Overlap: min(128, chunk_size // 2) so overlap < chunk_size
EX8_QUERY = "What is the complete procedure for draining and replacing the oil in the crankcase?"

results = []
for chunk_size in EX8_CHUNK_SIZES:
    overlap = min(128, chunk_size // 2)
    print(f"\n--- Chunk size = {chunk_size}, overlap = {overlap} ---")
    rebuild_pipeline(chunk_size=chunk_size, chunk_overlap=overlap)
    n_chunks = len(all_chunks)
    retrieved = retrieve(EX8_QUERY, top_k=5)
    answer = rag_query(EX8_QUERY, top_k=5)
    results.append({
        "chunk_size": chunk_size,
        "overlap": overlap,
        "n_chunks": n_chunks,
        "retrieved": retrieved,
        "answer": answer,
    })

# Save
lines = ["# Exercise 8: Chunk Size", f"# Corpus: {DOC_FOLDER}", ""]
lines.append(f"Query: {EX8_QUERY}\n")
for r in results:
    lines.append(f"\n## Chunk size = {r['chunk_size']} (overlap={r['overlap']}, n_chunks={r['n_chunks']})")
    lines.append("")
    for i, (chunk, score) in enumerate(r["retrieved"], 1):
        lines.append(f"  [{i}] Score: {score:.4f} | {chunk.source_file}")
        lines.append(f"      {chunk.text[:120]}...")
    lines.append("\nAnswer:")
    lines.append(r["answer"][:600] + ("..." if len(r["answer"]) > 600 else ""))
    lines.append("\n---")
with open(f"{EX8_OUTPUT}/chunk_size_results.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print(f"\n✓ Saved to {EX8_OUTPUT}/chunk_size_results.md")


--- Chunk size = 128, overlap = 64 ---


Batches:   0%|          | 0/602 [00:00<?, ?it/s]

Rebuilt: 19254 chunks, chunk_size=128, chunk_overlap=64

--- Chunk size = 256, overlap = 128 ---


Batches:   0%|          | 0/323 [00:00<?, ?it/s]

Rebuilt: 10305 chunks, chunk_size=256, chunk_overlap=128

--- Chunk size = 512, overlap = 128 ---


...

Rebuilt: 3277 chunks, chunk_size=512, chunk_overlap=128

--- Chunk size = 1024, overlap = 128 ---


Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Rebuilt: 1333 chunks, chunk_size=1024, chunk_overlap=128

--- Chunk size = 2048, overlap = 128 ---


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Rebuilt: 645 chunks, chunk_size=2048, chunk_overlap=128

✓ Saved to outputs/exercise8/chunk_size_results.md


In [162]:
# Create observations template for Exercise 8
obs = """# Exercise 8 Observations

## Top-5 retrieved chunks by chunk size

See chunk_size_results.md for full output.

## Index size (n_chunks) by chunk size

| Chunk size | Overlap | n_chunks | Notes |
|------------|---------|----------|-------|
| 128 | | | |
| 256 | | | |
| 512 | | | |
| 1024 | | | |
| 2048 | | | |

## Similarity scores (top chunk)

| Chunk size | Top score | Notes |
|------------|-----------|-------|
| 128 | | |
| 256 | | |
| 512 | | |
| 1024 | | |
| 2048 | | |

## Does larger chunk size improve retrieval of complete information?

[Compare answer completeness across chunk sizes. Do larger chunks capture full procedures better?]

## Cost: Index size, context granularity

[Fewer chunks with larger size? Trade-off: granularity vs. context per chunk?]

## Point of diminishing returns?

[At what chunk size did quality plateau?]

## Implications for chunking strategy

[What does this tell you about choosing chunk size for technical manuals?]

## Summary

- [Key takeaways]
"""
with open("outputs/exercise8/observations.md", "w", encoding="utf-8") as f:
    f.write(obs)
print("✓ Created outputs/exercise8/observations.md — fill in after reviewing results.")

✓ Created outputs/exercise8/observations.md — fill in after reviewing results.


---
## Exercise 9: Retrieval Score Analysis

Run 10 queries, retrieve top-10 chunks each, analyze score distribution (min, max, mean, spread). No pipeline rebuild — uses existing index.

In [164]:
# =============================================================================
# EXERCISE 9: Retrieval score analysis — 10 queries, top-10 chunks, score distribution
# =============================================================================
# Uses existing index (no rebuild). Saves to outputs/exercise9/
# =============================================================================

import os
import statistics
EX9_OUTPUT = "outputs/exercise9"
os.makedirs(EX9_OUTPUT, exist_ok=True)

EX9_QUERIES = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
    "What is the complete procedure for draining and replacing the oil in the crankcase?",
    "What is the recommended maintenance schedule for the engine?",
    "How do I check the magneto strength?",
    "What is the procedure for engine overhaul?",
    "How do I remove and replace the rear axle?",
    "What causes a knocking sound in the engine?",
]

results = {}
if index.ntotal > 0:
    for q in EX9_QUERIES:
        retrieved = retrieve(q, top_k=10)
        scores = [s for _, s in retrieved]
        results[q] = {"retrieved": retrieved, "scores": scores}
        print(f"Query: {q[:50]}... | top score: {scores[0]:.4f}" if len(q) > 50 else f"Query: {q} | top score: {scores[0]:.4f}")

    # Save detailed results
    lines = ["# Exercise 9: Retrieval Score Analysis", f"# Corpus: {DOC_FOLDER}", ""]
    all_scores = []
    for q, data in results.items():
        scores = data["scores"]
        all_scores.extend(scores)
        lines.append(f"\n## Query: {q}")
        lines.append(f"Top-10 scores: {[f'{s:.4f}' for s in scores]}")
        lines.append(f"Min: {min(scores):.4f} | Max: {max(scores):.4f} | Mean: {statistics.mean(scores):.4f} | Std: {(statistics.stdev(scores) if len(scores) > 1 else 0):.4f}")
        for i, (chunk, score) in enumerate(data["retrieved"], 1):
            lines.append(f"  [{i}] {score:.4f} | {chunk.source_file}")
        lines.append("")

    # Summary stats
    lines.append("\n## Overall score distribution (all 100 scores)")
    lines.append(f"Min: {min(all_scores):.4f} | Max: {max(all_scores):.4f} | Mean: {statistics.mean(all_scores):.4f} | Std: {statistics.stdev(all_scores):.4f}")
    lines.append("\n## Per-query summary")
    lines.append("| Query | Top | Min | Max | Mean | Std |")
    lines.append("|-------|-----|-----|-----|------|-----|")
    for q, data in results.items():
        s = data["scores"]
        std = statistics.stdev(s) if len(s) > 1 else 0
        q_safe = q.replace("|", " ")[:50]
        lines.append(f"| {q_safe} | {s[0]:.4f} | {min(s):.4f} | {max(s):.4f} | {statistics.mean(s):.4f} | {std:.4f} |")

    with open(f"{EX9_OUTPUT}/retrieval_scores.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"\n✓ Saved to {EX9_OUTPUT}/retrieval_scores.md")
else:
    print("Index empty. Load corpus and run pipeline first.")

Query: How do I adjust the carburetor on a Model T? | top score: 0.6392
Query: What is the correct spark plug gap for a Model T F... | top score: 0.5511
Query: How do I fix a slipping transmission band? | top score: 0.5404
Query: What oil should I use in a Model T engine? | top score: 0.4612
Query: What is the complete procedure for draining and re... | top score: 0.5825
Query: What is the recommended maintenance schedule for t... | top score: 0.4551
Query: How do I check the magneto strength? | top score: 0.6468
Query: What is the procedure for engine overhaul? | top score: 0.5486
Query: How do I remove and replace the rear axle? | top score: 0.5547
Query: What causes a knocking sound in the engine? | top score: 0.6738

✓ Saved to outputs/exercise9/retrieval_scores.md


In [165]:
# Create observations template for Exercise 9
obs = """# Exercise 9 Observations

## Top-10 retrieved chunks and scores by query

See retrieval_scores.md for full output.

## Score distribution summary

| Query | Top score | Min | Max | Mean | Std |
|-------|-----------|-----|-----|------|-----|
| (fill from retrieval_scores.md) | | | | | |

## Which queries had the highest/lowest top scores?

[Compare top scores across queries — which retrieved most relevant chunks?]

## Score spread (top vs bottom of top-10)

[Large spread = steep drop-off; small spread = similar relevance across top chunks]

## Implications for retrieval quality

[What does the score distribution tell you about retrieval confidence? Thresholds?]

## Summary

- [Key takeaways]
"""
with open("outputs/exercise9/observations.md", "w", encoding="utf-8") as f:
    f.write(obs)
print("✓ Created outputs/exercise9/observations.md — fill in after reviewing results.")

✓ Created outputs/exercise9/observations.md — fill in after reviewing results.


---
## Exercise 10: Prompt Template Variations

Test 5 prompt templates: minimal, strict grounding, citation, permissive, structured. Compare answers on the same queries.

In [166]:
# =============================================================================
# EXERCISE 10: Prompt template variations
# =============================================================================
# 5 templates: minimal, strict, citation, permissive, structured
# 3 queries. Saves to outputs/exercise10/
# =============================================================================

import os
EX10_OUTPUT = "outputs/exercise10"
os.makedirs(EX10_OUTPUT, exist_ok=True)

PROMPT_MINIMAL = """CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

PROMPT_STRICT_GROUNDING = """You are a helpful assistant. Answer using ONLY the context below. Do NOT use external knowledge.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS: If the context doesn't contain the answer, say "I cannot answer from the available documents." Otherwise, answer based solely on the context.

ANSWER:"""

PROMPT_CITATION = """You are a technical assistant. Answer the question using the context below. You MUST cite your sources.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS: For each fact, cite the source (e.g., "According to ModelTNew.txt..." or "Paragraph X states..."). If the context doesn't contain the answer, say so.

ANSWER:"""

PROMPT_PERMISSIVE = """Answer the question below. Use the provided context when relevant. If the context doesn't have enough information, you may use general knowledge to supplement, but prefer the context.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

PROMPT_STRUCTURED = """Answer the question using the context below. Format your answer as a clear, structured response.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS: Use numbered steps for procedures, or bullet points for lists. Be concise. If the context doesn't contain the answer, say so.

ANSWER:"""

EX10_TEMPLATES = [
    ("minimal", PROMPT_MINIMAL),
    ("strict_grounding", PROMPT_STRICT_GROUNDING),
    ("citation", PROMPT_CITATION),
    ("permissive", PROMPT_PERMISSIVE),
    ("structured", PROMPT_STRUCTURED),
]
EX10_QUERIES = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "What oil should I use in a Model T engine?",
]

if index.ntotal > 0:
    lines = ["# Exercise 10: Prompt Template Variations", f"# Corpus: {DOC_FOLDER}", ""]
    for q in EX10_QUERIES:
        lines.append(f"\n## Query: {q}\n")
        for tname, template in EX10_TEMPLATES:
            lines.append(f"### {tname}\n")
            ans = rag_query(q, top_k=5, show_context=False, prompt_template=template)
            lines.append(ans[:800] + ("..." if len(ans) > 800 else ""))
            lines.append("\n")
        lines.append("---\n")
    with open(f"{EX10_OUTPUT}/prompt_results.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"✓ Saved to {EX10_OUTPUT}/prompt_results.md")
else:
    print("Index empty. Load corpus and run pipeline first.")

✓ Saved to outputs/exercise10/prompt_results.md


In [167]:
# Create observations template for Exercise 10
obs = """# Exercise 10 Observations

## Answers by template

See prompt_results.md for full output.

## Which template produced the most accurate answers?

[Compare accuracy across templates for each query]

## Which template best avoided hallucination?

[Strict vs permissive — did permissive add wrong info?]

## Did citation improve traceability?

[Did the citation template produce usable source references?]

## Did structured format improve usability?

[Numbered steps, bullets — easier to follow?]

## Implications for prompt design

[What does this tell you about choosing prompt templates for RAG?]

## Summary

- [Key takeaways]
"""
with open("outputs/exercise10/observations.md", "w", encoding="utf-8") as f:
    f.write(obs)
print("✓ Created outputs/exercise10/observations.md — fill in after reviewing results.")

✓ Created outputs/exercise10/observations.md — fill in after reviewing results.


---
## Exercise 11 Cross-Document Synthesis

Test questions that require combining information from multiple chunks. Experiment with top_k (3, 5, 10). Does retrieving more chunks improve synthesis?

In [169]:
# =============================================================================
# EXERCISE 11 Cross-Document Synthesis
# =============================================================================
# Queries requiring info from multiple chunks. Test top_k = 3, 5, 10.
# Saves to outputs/exercise11/
# =============================================================================

import os
EX11_OUTPUT = "outputs/exercise11"
os.makedirs(EX11_OUTPUT, exist_ok=True)

# Synthesis queries — require combining info from multiple chunks
EX11_QUERIES = [
    "What are ALL the maintenance tasks I need to do monthly?",
    "Compare the procedures for adjusting the carburetor vs adjusting the transmission band.",
    "What tools do I need for a complete tune-up?",
    "Summarize all safety warnings in the manual.",
]
EX11_TOP_K = [3, 5, 10]

if index.ntotal > 0:
    lines = ["# Exercise 11: Cross-Document Synthesis", f"# Corpus: {DOC_FOLDER}", ""]
    for q in EX11_QUERIES:
        lines.append(f"\n## Query: {q}\n")
        for k in EX11_TOP_K:
            lines.append(f"### top_k = {k}\n")
            ans = rag_query(q, top_k=k, show_context=False)
            lines.append(ans[:700] + ("..." if len(ans) > 700 else ""))
            lines.append("\n")
        lines.append("---\n")
    with open(f"{EX11_OUTPUT}/synthesis_results.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"✓ Saved to {EX11_OUTPUT}/synthesis_results.md")
else:
    print("Index empty. Load corpus and run pipeline first.")

✓ Saved to outputs/exercise11/synthesis_results.md


In [170]:
# Create observations template for Exercise 11
obs = """# Exercise 11 observations

## Synthesis results by top_k

See synthesis_results.md for full output.

## Can the model successfully combine information from multiple chunks?

[Compare answers across top_k — did k=10 produce more complete synthesis than k=3?]

## Does retrieving more chunks improve synthesis?

[Does higher top_k lead to more comprehensive answers? Or more noise?]

## Does it miss information that wasn't retrieved?

[For "ALL maintenance tasks" or "all safety warnings" — did the model acknowledge gaps when k was low?]

## Does contradictory information in different chunks cause problems?

[Did the model handle conflicting info (e.g., different procedures) or conflate them?]

## Implications for synthesis queries

[What does this suggest for top_k and prompt design when synthesis is needed?]

## Summary

- [Key takeaways]
"""
with open("outputs/exercise11/observations.md", "w", encoding="utf-8") as f:
    f.write(obs)
print("✓ Created outputs/exercise11/observations.md — fill in after reviewing results.")

✓ Created outputs/exercise11/observations.md — fill in after reviewing results.


In [125]:
# EXPERIMENT 1: Compare WITH vs WITHOUT RAG
# ==========================================

question = "What are the specifications for the landing gear?"  # ← Use a corpus-specific question!

if index.ntotal > 0:
    # WITHOUT RAG - just ask the model directly
    direct_prompt = f"""Answer this question:
{question}

Answer:"""
    
    print("WITHOUT RAG (model's own knowledge):")
    print("-" * 40)
    direct_answer = generate_response(direct_prompt)
    print(direct_answer)
    
    print("\n" + "=" * 60 + "\n")
    
    # WITH RAG
    print("WITH RAG (using retrieved context):")
    print("-" * 40)
    rag_answer = rag_query(question, top_k=5)
    print(rag_answer)
else:
    print("Please complete the pipeline setup first.")

WITHOUT RAG (model's own knowledge):
----------------------------------------
The landing gear is a set of wheels that extend from an aircraft to provide ground contact during takeoff and landing. It includes the main landing gear, nose wheel steering mechanism, and tail skid or stabilizer.
The landing gear consists of two sets of wheels (main landing gears) mounted on the fuselage of the aircraft. These wheels support the weight of the aircraft when it lands and take off. They also allow the aircraft to move sideways while taxiing on the runway.

In addition to the main landing gears, some aircraft have a nose wheel steering mechanism which allows pilots to steer the aircraft without moving the nose wheel itself. This helps in maneuvering the aircraft through tight spaces such as airport runways.

Some modern aircraft also feature a tail skid or stabilizer, which provides additional stability during takeoff and landing by acting like a small wing at the rear of the aircraft. This can 

In [126]:
# EXPERIMENT 2: Effect of top_k
# ==========================================

question = "What safety procedures are required?"  # ← Use a corpus-specific question!

if index.ntotal > 0:
    for k in [1, 3, 5, 10]:
        print(f"\n{'='*60}")
        print(f"TOP_K = {k}")
        print(f"{'='*60}")
        answer = rag_query(question, top_k=k)
        print(answer[:500] + "..." if len(answer) > 500 else answer)
else:
    print("Please complete the pipeline setup first.")


TOP_K = 1
The safety reviews conducted under paragraph (2) must be detailed in reports submitted by the Administrator to the appropriate congressional committees every six months following the initial enactment of this section. This ensures transparency and accountability regarding the outcomes of these safety evaluations. 

The specific details of what constitutes "safety procedures" or how they are implemented are not explicitly stated in the given context. However, it is implied that the safety reviews...

TOP_K = 3
The safety procedures require the Administrator to conduct safety reviews every six months after the enactment of this section and submit reports detailing the analyses and results of these reviews to the appropriate committees of Congress. Additionally, it mandates that flight operations conducted by the Department of Defense, emergency response providers, and air medical transport operators be evaluated to assess any associated safety risks to commercial transport air

In [127]:
# EXPERIMENT 3: Question the corpus CAN'T answer
# ==========================================
# Does the model admit it doesn't know, or hallucinate?

unanswerable_question = "What is the CEO's favorite color?"

if index.ntotal > 0:
    print(f"Question: {unanswerable_question}\n")
    answer = rag_query(unanswerable_question, top_k=5, show_context=True)
    print(f"\nAnswer: {answer}")
else:
    print("Please complete the pipeline setup first.")

Question: What is the CEO's favorite color?

RETRIEVED CONTEXT:
[Source: CREC-2026-01-14.txt, Relevance: 0.441]
pect.
I also particularly thank my good
friend and working partner, the distinguished ranking member of the full
committee, Ms. DELAURO, and I also
thank the superb staff on both sides of
the aisle who worked tirelessly to
present us with the product befor

---

[Source: CREC-2026-01-28.txt, Relevance: 0.417]
red.
RECOGNITION OF THE MINORITY LEADER

The Democratic leader is recognized.
DEPARTMENT OF HOMELAND SECURITY

Mr. SCHUMER. Mr. President, it has
been 4 days since Alex Pretti was murdered in broad daylight by Federal
agents in Minneapolis, and there has
b

---

[Source: CREC-2026-01-14.txt, Relevance: 0.390]
Strong capital markets are essential to American economic leadership,
and ensuring that the SEC has the resources, clear rules, and effective oversight to carry out its mission is essential.
Mr. Chairman, I thank the chairman
for his leadership.

---

[Source: CREC-

---
## Save/Load Your Index

For large corpora, you don't want to re-embed every time. Here's how to persist the index.

In [128]:
import pickle

def save_index(filepath: str):
    """Save FAISS index and chunks to disk."""
    faiss.write_index(index, f"{filepath}.faiss")
    with open(f"{filepath}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved index to {filepath}.faiss")
    print(f"✓ Saved chunks to {filepath}.chunks")

def load_saved_index(filepath: str):
    """Load FAISS index and chunks from disk."""
    global index, all_chunks
    index = faiss.read_index(f"{filepath}.faiss")
    with open(f"{filepath}.chunks", 'rb') as f:
        all_chunks = pickle.load(f)
    print(f"✓ Loaded index with {index.ntotal} vectors")

# Save your index
if index.ntotal > 0:
    save_index("my_rag_index")
else:
    print("No index to save.")

# Later, to load:
# load_saved_index("my_rag_index")

✓ Saved index to my_rag_index.faiss
✓ Saved chunks to my_rag_index.chunks


---
## Next Steps

You've built a complete RAG pipeline from scratch! In the next class, we'll:

1. **Improve retrieval** with query rewriting and hybrid search
2. **Rebuild with LangChain** to see how frameworks abstract these steps
3. **Evaluate systematically** with test questions and metrics

### Exercises to try:
- Vary chunk size (256, 512, 1024) and measure retrieval quality
- Try a different embedding model (`BAAI/bge-small-en-v1.5`)
- Try a larger LLM (`Qwen/Qwen2.5-3B-Instruct`) and compare answer quality
- Ask questions that require combining information from multiple chunks

---
## Appendix: Device Information

Run this cell to see detailed information about your compute environment.

In [171]:
def print_device_info():
    """Print detailed information about available compute devices."""
    print("=" * 60)
    print("DEVICE INFORMATION")
    print("=" * 60)
    
    print(f"\nEnvironment: {ENVIRONMENT}")
    print(f"PyTorch version: {torch.__version__}")
    
    # CUDA
    print(f"\nCUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  Device: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # MPS
    print(f"\nMPS available: {torch.backends.mps.is_available()}")
    print(f"MPS built: {torch.backends.mps.is_built()}")
    
    # Current selection
    print(f"\n→ Selected device: {DEVICE}")
    print(f"→ Selected dtype: {DTYPE}")
    print("=" * 60)

print_device_info()

DEVICE INFORMATION

Environment: local
PyTorch version: 2.10.0+cu128

CUDA available: True
  Device: NVIDIA RTX A6000
  Memory: 51.0 GB

MPS available: False
MPS built: False

→ Selected device: cuda
→ Selected dtype: torch.float16
